In [1]:
import Pkg; Pkg.add(["Ipopt", "SpecialFunctions", "DataFrames"])


   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [2]:
using Random, Distributions, JuMP, Ipopt, SpecialFunctions, DataFrames


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor $L$, value function $V$, migration shares $\mu$, mobility costs $\tau$) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production ($A$, $w$, $\kappa$, $\theta$, $\eta$, $\gamma$, $\alpha$) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$.

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ — discount factor
- $\theta^j$ — Fréchet trade elasticity in sector $j$
- $\nu$ — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$, the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [3]:
#try indexing format of J[region]_[sector]_[time], x if not indexed by that component

N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1, i.e. sector 0) plus
          # the J real sectors (market columns 2:M, i.e. sectors 1:J). Per the model, a "market"
          # is a region-sector pair (n,j) with j = 0,...,J, so any array indexed over "every
          # market a household could be in" (L, V, mu, tau_mig) has rows = regions (N), columns
          # = markets (M), with column 1 = non-employment. Arrays that only pertain to real
          # production (A, w, kappa, theta, eta, gamma, alpha) keep rows = regions, columns =
          # real sectors (J) -- to look one of these up against a market-indexed array, offset
          # the sector index by +1 (real sector j lives in market column j+1).
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) #labor force in economy at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # Seed for the random number generator (to guarantee reproducibility; this is standard in research these days)

A_0 = rand(N,J) #region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) #arbitrary coefficient

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector (kappa: trade cost)

# sigma = 2 # Substitution elasticity between goods
# L = ones(N, 1) # Size of labor force in each country

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k), j,k = 0,...,J (market columns 1:M,
# where column 1 is non-employment); 0 to stay in the same market, 1 otherwise (tau: migration cost)
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


10

## Baseline levels

Everything from here on is expressed in *changes* relative to a baseline allocation -- so we
first need an actual baseline. As in Notebook 3, we solve the Temporary Equilibrium (Definition 1)
at $L_0$ to get market-clearing wages $w_{temp}$ and trade shares $\pi_{temp}$; these play the
role of $(w_t, \pi_t)$ in the hat-algebra equations below.

In [4]:
# Temporary equilibrium (Definition 1, PDF Section 5): given labor supply L and the fundamentals
# (A_0, kappa_0), find wages w = {w^{nj}} that clear goods and labor markets simultaneously --
# eq 6 (goods clearing / expenditure) and eq 7 (labor clearing) together, N x J equations in N x J
# wage unknowns. This actually SOLVES for w (via JuMP + Ipopt, posed as a feasibility problem:
# minimize a trivial 0 objective subject to the market-clearing equations as constraints).
#
# The system is homogeneous of degree 1 in w: scaling every wage by a constant lambda scales the
# wage bill (LHS of eq 7) by lambda, and scales expenditure X (eq 6) by lambda too, while trade
# shares pi (eq 5) are unaffected (they only depend on RELATIVE wages across sources). So only
# relative wages are pinned down -- we normalize w[1,1] = 1 and drop that market's own clearing
# equation, which is redundant with the rest by Walras' law (aggregate labor income always equals
# aggregate expenditure here, since there is no trade deficit).

function trade_shares_and_expenditure(w::AbstractMatrix, L::AbstractMatrix)
    x = B .* w # unit cost (eq 4.2)
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    pi = [trade_cost_term[n,j,i] / sum(trade_cost_term[n,j,:]) for n in 1:N, j in 1:J, i in 1:N] # eq 5
    I = vec(sum(w .* L[:, 2:M], dims=2)) # labor income by region (employed markets only, eq analog of I_0 above)
    X = I * alpha' # eq 6
    return pi, X
end

function solve_temporary_equilibrium(L::AbstractMatrix; w_guess = ones(N,J))
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    @variable(model, w[n=1:N, j=1:J] >= 1e-6, start = w_guess[n,j])

    @expression(model, tct[n=1:N, j=1:J, i=1:N],
        (B[i,j]*w[i,j]*kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j]))
    @expression(model, pishare[n=1:N, j=1:J, i=1:N], tct[n,j,i] / sum(tct[n,j,m] for m in 1:N))
    @expression(model, Inc[n=1:N], sum(w[n,k]*L[n,k+1] for k in 1:J))
    @expression(model, X[n=1:N, j=1:J], alpha[j]*Inc[n])

    @constraint(model, w[1,1] == 1.0) # numeraire, replaces the redundant (1,1) equation
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w[n,j]*L[n,j+1] == sum(pishare[i,j,n]*X[i,j] for i in 1:N)) # eq 7
        end
    end

    @objective(model, Min, 0) # feasibility problem: no objective, just satisfy the constraints
    optimize!(model)

    w_star = value.(w)
    pi_star, X_star = trade_shares_and_expenditure(w_star, L)
    return w_star, pi_star, X_star, termination_status(model)
end


solve_temporary_equilibrium (generic function with 1 method)

In [5]:
# Solve the temporary equilibrium at the initial labor distribution L_0 -- this is the
# baseline (w_temp, pi_temp) that the hat-algebra system below is defined relative to.

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)
println("Solver status: ", status_temp)



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Solver status: LOCALLY_SOLVED


In [6]:
# stationary_V(U_mkt): the Bellman fixed point (Notebook 1's Migration Decision section /
# Notebook 3's Sequential Equilibrium section), as a function of flow utility U_mkt -- needed
# here to get the baseline mu_stationary below (the migration shares of the economy's own
# stationary equilibrium under w_temp).
function stationary_V(U_mkt::AbstractMatrix; tol=1e-12, maxiter=10_000)
    V = log.(U_mkt)
    for _ in 1:maxiter
        V_next = [
            log(U_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
            for n in 1:N, j in 1:M
        ]
        if maximum(abs.(V_next .- V)) < tol
            V = V_next
            break
        end
        V = V_next
    end
    return V
end

# migration_shares(V_next): the logit migration shares (eq 3), as a function of next period's
# value function.
function migration_shares(V_next::AbstractMatrix)
    [
        exp((beta*V_next[i,k] - tau_mig[n,j,i,k]) / nu) /
        sum(exp((beta*V_next[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end


migration_shares (generic function with 1 method)

# Dynamic Hat Algebra

## Temporary equilibrium in time differences

The payoff of this method: given an observed baseline allocation at time $t$ — labor $L_t$, trade shares $\pi_t$, wages $w_t$ — and a *change* in labor supply $\dot L_{t+1}=L_{t+1}/L_t$ and fundamentals $\dot A_{t+1}, \dot\kappa_{t+1}$, we can solve for $\dot w_{t+1}=w_{t+1}/w_t$ (and hence $\dot P_{t+1}$, the new trade shares, and the new expenditure levels) **without ever knowing the level of $A_t$ or $\kappa_t$** — only their proportional change.

$$\dot x_{t+1}^{nj} = \dot w_{t+1}^{nj} \tag{8}$$
$$\dot P_{t+1}^{nj} = \left(\sum_{i=1}^N \pi_t^{nj,ij}\left(\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j}\right)^{-1/\theta^j} \tag{9}$$
$$\pi_{t+1}^{nj,ij} = \pi_t^{nj,ij}\left(\frac{\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}}{\dot P_{t+1}^{nj}}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j} \tag{10}$$
$$X_{t+1}^{nj} = \alpha^j\sum_{k=1}^J \dot w_{t+1}^{nk}\dot L_{t+1}^{nk}\, w_t^{nk}L_t^{nk} \tag{11}$$
$$\dot w_{t+1}^{nj}\dot L_{t+1}^{nj}\, w_t^{nj}L_t^{nj} = \sum_{i=1}^N \pi_{t+1}^{ij,nj} X_{t+1}^{ij} \tag{12}$$

A subtlety worth flagging: despite eq. (9) producing a genuine *ratio* $\dot P_{t+1}=P_{t+1}/P_t$, eq. (10) already multiplies in the baseline $\pi_t$, so what it produces is the actual **level** $\pi_{t+1}$ (not a further ratio) — that's exactly what eq. (12) needs, since it's a level equation (nominal revenue at $t+1$ on both sides). Similarly $X_{t+1}^{nj}$ from eq. (11) is a level (it's built entirely from $t$-levels and the given changes, and — unlike the full model — never needs to self-reference other $X_{t+1}$'s, since there's no materials network to close the loop through).

Only $\dot w_{t+1}$ is unknown; everything else here is a direct function of it plus the given baseline and shock. As in Notebook 3's Temporary Equilibrium section, the system is homogeneous of degree 1 in $\dot w_{t+1}$ (nominal wage growth is only pinned down up to a numeraire — trade shares and real quantities are invariant to it), so we again normalize $\dot w_{t+1}^{11}=1$ and drop that market's own equation.

In [7]:
# Temporary equilibrium in time differences (eqs 8-12): given a baseline allocation at t
# (pi_t, w_t, L_t) and a change in labor supply / fundamentals (L_dot, A_dot, kappa_dot), solve
# for w_dot = w_{t+1}/w_t. kappa_dot has the same shape as kappa_0: a length-J vector of N x N
# matrices. L_dot is N x M (market space, column 1 = non-employment); only its real-sector
# columns (2:M) enter here, offset by +1 as usual.
#
# pi_next (eq 10) and X_next (eq 11) come out as LEVELS despite living inside the hat system --
# pi_next already has the baseline pi_t multiplied in, and X_next is built from t-level income
# terms -- so eq 12 (a level equation: nominal revenue at t+1 on both sides) uses them directly,
# with no further multiplication by pi_t needed.

function solve_temp_eq_hat(pi_t::Array{Float64,3}, w_t::AbstractMatrix, L_t::AbstractMatrix,
                            L_dot::AbstractMatrix, A_dot::AbstractMatrix, kappa_dot;
                            w_dot_guess = ones(N,J))
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    @variable(model, w_dot[n=1:N, j=1:J] >= 1e-6, start = w_dot_guess[n,j])

    @expression(model, P_dot[n=1:N, j=1:J],
        (sum(pi_t[n,j,i] * (w_dot[i,j]*kappa_dot[j][n,i])^(-theta[j]) * A_dot[i,j]^theta[j] for i in 1:N))^(-1/theta[j])) # eq 9
    @expression(model, pi_next[n=1:N, j=1:J, i=1:N],
        pi_t[n,j,i] * ((w_dot[i,j]*kappa_dot[j][n,i]) / P_dot[n,j])^(-theta[j]) * A_dot[i,j]^theta[j]) # eq 10
    @expression(model, X_next[n=1:N, j=1:J],
        alpha[j] * sum(w_dot[n,k]*L_dot[n,k+1]*w_t[n,k]*L_t[n,k+1] for k in 1:J)) # eq 11

    @constraint(model, w_dot[1,1] == 1.0) # numeraire
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w_dot[n,j]*L_dot[n,j+1]*w_t[n,j]*L_t[n,j+1] == sum(pi_next[i,j,n]*X_next[i,j] for i in 1:N)) # eq 12
        end
    end

    @objective(model, Min, 0)
    optimize!(model)

    return value.(w_dot), value.(P_dot), value.(pi_next), value.(X_next), termination_status(model)
end


solve_temp_eq_hat (generic function with 1 method)

## Sequential equilibrium in time differences (household block)

This half of the system is untouched by the production-side simplification -- it never depended on structures or materials -- so it's identical in spirit to the Migration Decision section above, just restated in changes:

$$\mu_{t+1}^{nj,ik} = \frac{\mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}}{\sum_{m=1}^N\sum_{h=0}^J \mu_t^{nj,mh}\left(\dot u_{t+2}^{mh}\right)^{\beta/\nu}} \tag{13}$$
$$\dot u_{t+1}^{nj} = \dot\omega^{nj}(\dot L_{t+1},\dot\Theta_{t+1})\left(\sum_{i=1}^N\sum_{k=0}^J \mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}\right)^{\nu} \tag{14}$$
$$L_{t+1}^{nj} = \sum_{i=1}^N\sum_{k=0}^J \mu_t^{ik,nj}L_t^{ik} \tag{15}$$

where $u_t^{nj}\equiv\exp(V_t^{nj})$, and the connecting piece is the *real* wage change $\dot\omega_{t+1}^{nj}=\dot w_{t+1}^{nj}/\dot P_{t+1}^n$ (with $\dot P_{t+1}^n=\prod_k(\dot P_{t+1}^{nk})^{\alpha^k}$, and $\dot\omega_{t+1}^{n0}=1$ always, since $b^n$ never changes). Despite the dot, eq. (13)'s left-hand side is a **level** — it already has $\mu_t$ multiplied in — so it directly gives $\mu_{t+1}$, not a further ratio to apply on top of $\mu_t$.

As in the Migration Decision section, we specialize to the **stationary case**: a shock that permanently shifts $\dot\omega$ to a new constant value, so $\dot u_{t+1}=\dot u_{t+2}=\dot u^*$ for every $t$, solved by the same kind of fixed-point iteration used there — just multiplicatively in $\dot u$ instead of additively in $V$.

In [8]:
# Household block in time differences, stationary case (eqs 13-15). u_dot is the analogue of the
# V fixed point in Notebook 1's Migration Decision section, but multiplicative: u_dot^{nj} = omega_dot^{nj}
# * (sum_{i,k} mu_baseline^{nj,ik} * (u_dot^{ik})^{beta/nu})^nu. mu_baseline is the ORIGIN period's
# (level) migration shares -- here, mu_0 from Notebook 1's Migration Decision section -- used as fixed
# weights, exactly as pi_t was used as fixed weights in P_dot (eq 9) above.

function stationary_u_hat(omega_dot::AbstractMatrix, mu_baseline::Array{Float64,4}; tol=1e-12, maxiter=10_000)
    u_dot = ones(N, M)
    for _ in 1:maxiter
        u_dot_next = [
            omega_dot[n,j] * (sum(mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
            for n in 1:N, j in 1:M
        ]
        converged = maximum(abs.(u_dot_next .- u_dot)) < tol
        u_dot = u_dot_next
        converged && break
    end
    return u_dot
end

# eq 13: mu_{t+1} is a LEVEL despite living in the hat system (mu_baseline is already multiplied
# in), exactly parallel to pi_next above.
function migration_shares_next(u_dot::AbstractMatrix, mu_baseline::Array{Float64,4})
    [
        mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) /
        sum(mu_baseline[n,j,m,h] * u_dot[m,h]^(beta/nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end

# flow_utility_mkt, parametrized by A/kappa (a companion to flow_utility_mkt from Notebook 3's
# Sequential Equilibrium section, computing flow utility at an explicit fundamentals LEVEL rather than
# reading the global A_0/kappa_0 -- needed by the Sandbox section below to get a baseline mu
# under the market-clearing wage w_temp rather than the toy placeholder w_0).

function flow_utility_mkt_at(w::AbstractMatrix, A::AbstractMatrix, kappa)
    x = B .* w
    trade_cost_term = [
        (x[i,j] * kappa[j][n,i])^(-theta[j]) * A[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    Gamma = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]
    P = [Gamma[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j]) for n in 1:N, j in 1:J]
    P_hat_region = [prod((P[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N]
    C = [w[n,j] / P_hat_region[n] for n in 1:N, j in 1:J]
    return hcat(b, C)
end


flow_utility_mkt_at (generic function with 1 method)

# Solving for the Transition Path

Given a baseline allocation $(L_0,\pi_0,w_0)$, last period's realized migration shares $\mu_{-1}$, and a path of fundamental *changes* $\dot\Theta_t=(\dot A_t,\dot\kappa_t)$ for $t=1,\dots,T$ (beyond $T$ we assume no further change, i.e. $\dot\Theta_{T+1}=1$), this section computes the whole transition path $\{L_t,\mu_t,w_t\}$ using only the temporary-equilibrium-in-differences system (eqs 8-12 above) and the household-block-in-differences system (eqs 13-15 above) chained together period by period -- a **guess-forward-simulate-backward-solve** fixed-point procedure.

Unlike the earlier stationary-case validation (where $\dot\omega$ was assumed constant forever), here $\dot u_t$ genuinely varies with $t$ along the transition, so it can't be pinned down by a simple forward fixed-point iteration -- it has to be solved by *backward induction* against a *forward*-simulated $\mu$ path, and those two are coupled (the $\mu$ path needs $\dot u$; $\dot u$ needs the $\mu$ path), which is why the whole thing is an outer loop:

1. **Guess** a full path $\{\dot u_t\}_{t=1}^T$ (start at "no change," $\dot u_t=1$), with terminal condition $\dot u_{T+1}=1$.
2. **Forward-simulate migration shares** $\{\mu_t\}_{t=0}^T$ via eq. (13), seeded by $\mu_{-1}$: each $\mu_t$ needs $\mu_{t-1}$ and $\dot u_{t+1}$ (using the terminal $\dot u_{T+1}=1$ when $t=T$).
3. **Forward-simulate labor** $\{L_t\}_{t=1}^{T+1}$ via eq. (15) -- one period past $T$, since $\mu_T$ generates $L_{T+1}$, needed by the next step.
4. **Solve the temporary equilibrium period by period**, $t=0,\dots,T$: given $\dot L_{t+1}=L_{t+1}/L_t$ and the shock $\dot\Theta_{t+1}$, call `solve_temp_eq_hat` (eqs 8-12) against the *currently tracked* baseline $(\pi_t,w_t,L_t)$ -- not the period-0 baseline -- to get $\dot w_{t+1}$, then update $\pi_{t+1}=\pi_t\cdot(\text{eq 10 bracket})$ and $w_{t+1}=\dot w_{t+1}\cdot w_t$ and carry those forward as the new "current" levels for the next $t$. This is exactly why levels (not just growth rates) have to be tracked through the loop -- eq. (11)'s $X_{t+1}$ needs the actual wage-bill level $w_t^{nk}L_t^{nk}$, which compounds.
5. **Backward-solve** an updated path $\{\dot u_t\}_{t=1}^T$ via eq. (14): $\dot u_{t+1}$ is built from the lagged share $\mu_t$ (from step 2), the real-wage change $\dot\omega_{t+1}=\dot w_{t+1}/\dot P_{t+1}^n$ (from step 4), and the next period's $\dot u_{t+2}$ -- working backward from the fixed terminal $\dot u_{T+1}=1$.
6. **Compare** the updated path to the guess via $\max_t|\dot u_t^{(1)}-\dot u_t^{(0)}|<\varepsilon$.
7. **Iterate** (with damping) until converged.

As in the single-period case, `solve_temp_eq_hat`'s numeraire $\dot w_{t+1}^{1,1}\equiv1$ is applied fresh at *every* $t$ -- eq. (12) is homogeneous of degree zero in $\dot w_{t+1}$ within a period, so only relative wages are pinned down each period, never a cross-period price level.

In [9]:
# Indexing convention used throughout (Julia is 1-indexed, but the model's time index starts at
# 0 or -1 for several objects):
#   L_path[s]         = L_{s-1}      for s = 1,...,T+2   (L_path[1] = L_0, ..., L_path[T+2] = L_{T+1})
#   mu_path[s]         = mu_{s-1}     for s = 1,...,T+1   (mu_path[1] = mu_0, ..., mu_path[T+1] = mu_T)
#   omega_dot_path[s]  = omega_dot_s  for s = 1,...,T+1   (direct indexing, no shift)
#   u_dot_guess[s]     = u_dot_s      for s = 1,...,T     (direct indexing; u_dot_{T+1} = 1 is a
#                                                           separate fixed constant, not in this array)
#   A_dot_path[s], kappa_dot_path[s] = shock going INTO period s, i.e. Theta_dot_s = Theta_s/Theta_{s-1},
#                                      for s = 1,...,T (beyond T, Theta_dot is implicitly 1: no further change)

function solve_transition_path_hat(w_0::AbstractMatrix, L_0::AbstractMatrix, pi_0::Array{Float64,3},
                                    mu_minus1::Array{Float64,4}, A_dot_path::Vector, kappa_dot_path::Vector;
                                    T::Int, max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5)

    u_dot_guess = [ones(N,M) for _ in 1:T] # step 1: initial "no change" guess for u_dot_1,...,u_dot_T
    u_dot_terminal = ones(N,M)             # fixed terminal condition, u_dot_{T+1} = 1

    local mu_path, L_path, omega_dot_path, w_path, pi_final, converged, outer_used

    for outer in 1:max_outer
        # Step 2: forward-simulate mu_0,...,mu_T (eq 13), seeded by mu_minus1.
        # mu_t needs mu_{t-1} and u_dot_{t+1} -- for t=T that's the fixed terminal u_dot_{T+1}.
        mu_path = Vector{Array{Float64,4}}(undef, T+1)
        mu_prev = mu_minus1
        for t in 0:T
            u_dot_tp1 = (t+1 <= T) ? u_dot_guess[t+1] : u_dot_terminal
            mu_path[t+1] = migration_shares_next(u_dot_tp1, mu_prev)
            mu_prev = mu_path[t+1]
        end

        # Step 3: forward-simulate L_1,...,L_{T+1} (eq 15) -- one period past T, since mu_T
        # generates L_{T+1}, needed by step 4's last temporary-equilibrium solve (T -> T+1).
        L_path = Vector{Matrix{Float64}}(undef, T+2)
        L_path[1] = L_0
        for t in 0:T
            L_path[t+2] = [
                sum(mu_path[t+1][i,k,n,j] * L_path[t+1][i,k] for i in 1:N, k in 1:M)
                for n in 1:N, j in 1:M
            ]
        end

        # Step 4: solve the temporary equilibrium period by period, t=0,...,T, tracking the
        # LEVELS (pi_t, w_t) forward across periods (not just growth rates), since eq 11's
        # X_{t+1} needs the actual compounding wage-bill level.
        omega_dot_path = Vector{Matrix{Float64}}(undef, T+1)
        w_path = Vector{Matrix{Float64}}(undef, T+2)
        w_path[1] = w_0
        pi_current = pi_0
        for t in 0:T
            # Guard against 0/0: if a market's labor mass is (numerically) zero at t, its
            # L_dot doesn't matter -- the eq 12 constraint below multiplies it by L_t=0 anyway
            # -- but literal NaN would still poison that constraint's coefficient (0 * NaN =
            # NaN in IEEE floats, not 0). Use a finite placeholder (1.0) instead of dividing.
            L_dot_step = [
                L_path[t+1][n,j] == 0 ? 1.0 : L_path[t+2][n,j] / L_path[t+1][n,j]
                for n in 1:N, j in 1:M
            ]
            A_dot_step = (t+1 <= T) ? A_dot_path[t+1] : ones(N,J)
            kappa_dot_step = (t+1 <= T) ? kappa_dot_path[t+1] : [ones(N,N) for _ in 1:J]

            w_dot, P_dot, pi_next, X_next, status = solve_temp_eq_hat(
                pi_current, w_path[t+1], L_path[t+1], L_dot_step, A_dot_step, kappa_dot_step)

            P_hat_dot = [prod(P_dot[n,k]^alpha[k] for k in 1:J) for n in 1:N]
            omega_dot_path[t+1] = hcat(ones(N), w_dot ./ P_hat_dot)
            w_path[t+2] = w_dot .* w_path[t+1]
            pi_current = pi_next
        end
        pi_final = pi_current

        # Step 5: backward-solve u_dot_1,...,u_dot_T (eq 14), from the fixed terminal
        # u_dot_{T+1}=1 down to u_dot_1. u_dot_{t+1} needs mu_t (= mu_path[t+1]), omega_dot_{t+1}
        # (= omega_dot_path[t+1]), and u_dot_{t+2} (the previous, i.e. one-later, backward step).
        u_dot_new = Vector{Matrix{Float64}}(undef, T)
        u_dot_next = u_dot_terminal
        for t in (T-1):-1:0
            u_dot_new[t+1] = [
                omega_dot_path[t+1][n,j] * (sum(mu_path[t+1][n,j,i,k] * u_dot_next[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
                for n in 1:N, j in 1:M
            ]
            u_dot_next = u_dot_new[t+1]
        end

        # Steps 6-7: compare and iterate (damped).
        diff = maximum(maximum(abs.(u_dot_new[t] .- u_dot_guess[t])) for t in 1:T)
        u_dot_guess = [damp .* u_dot_new[t] .+ (1-damp) .* u_dot_guess[t] for t in 1:T]
        converged = diff < tol
        outer_used = outer
        if converged
            println("solve_transition_path_hat converged after $outer outer iterations (max u_dot change = $diff)")
            break
        end
        if outer == max_outer
            println("solve_transition_path_hat: reached max_outer=$max_outer without converging (max u_dot change = $diff)")
        end
    end

    return (; L_path, mu_path, w_path, pi_final, u_dot_path = u_dot_guess, omega_dot_path, converged, outer_used)
end


solve_transition_path_hat (generic function with 1 method)

# Logging Dynamic Hat Solver Runs

Every call to `solve_transition_path_hat` below (routed through `run_shock_scenario` in the Sandbox section) appends one row to `transition_runs`: the conditions the solver was given -- the baseline allocations $(w_0,L_0,\pi_0,\mu_{-1})$, the fundamentals path $(\dot A_t,\dot\kappa_t)$, and the solver settings $(T,\text{max\_outer},\text{tol},\text{damp})$ -- alongside the full transition path it returned, so any scenario can be compared or inspected later without re-running the solver.

In [10]:
# Every call to solve_transition_path_hat (made via run_shock_scenario in the Sandbox section
# below) appends one row here: the conditions the solver was fed (baseline allocations, the
# fundamentals path, and the solver's own settings) plus the full transition path it returned.

# migration_flows: from a scenario's mu_path/L_path (eqs 13/15), computes the gross and net
# migration flow into each market (n,j) at every period t=0,...,T. "Gross" is total churn through
# the market -- everyone who arrives from elsewhere PLUS everyone who leaves for elsewhere,
# excluding the (n,j)->(n,j) "stayers" term -- so it stays positive even once the economy has
# settled into a new steady state (idiosyncratic taste shocks keep people reshuffling), and is
# exactly zero only when mu_t^{nj,nj} = 1 for everyone (the no-mobility limit, tau_mig -> infty).
# "Net" is just L_{t+1}^{nj} - L_t^{nj} (inflow minus outflow), which -> 0 as the labor
# distribution converges to its new long-run allocation, regardless of how much churn remains.
function migration_flows(mu_path::Vector{Array{Float64,4}}, L_path::Vector{Matrix{Float64}})
    Tflow = length(mu_path) # one entry per t = 0,...,T, matching mu_path's indexing (path[s] = value at t=s-1)
    gross_path = Vector{Matrix{Float64}}(undef, Tflow)
    net_path = Vector{Matrix{Float64}}(undef, Tflow)
    for s in 1:Tflow
        mu_t, L_t, L_tp1 = mu_path[s], L_path[s], L_path[s+1]
        Nn, Mm = size(L_t)
        gross_in = [sum(mu_t[i,k,n,j]*L_t[i,k] for i in 1:Nn, k in 1:Mm if !(i==n && k==j)) for n in 1:Nn, j in 1:Mm]
        gross_out = [L_t[n,j]*(1 - mu_t[n,j,n,j]) for n in 1:Nn, j in 1:Mm]
        gross_path[s] = gross_in .+ gross_out
        net_path[s] = L_tp1 .- L_t
    end
    return gross_path, net_path
end

transition_runs = DataFrame(
    label = String[],
    A_shocks = Any[],       # productivity shock specs (n,j,factor,ramp) passed to run_shock_scenario
    kappa_shocks = Any[],   # trade-cost shock specs (n,j,i,factor,ramp) passed to run_shock_scenario
    w_0 = Any[],             # baseline wage fed to the solver (N x J)
    L_0 = Any[],             # baseline labor distribution fed to the solver (N x M)
    pi_0 = Any[],            # baseline trade shares fed to the solver (N x J x N)
    mu_minus1 = Any[],      # mu_{-1} fed to the solver (N x M x N x M)
    A_dot_path = Any[],     # path of productivity changes, t = 1,...,T
    kappa_dot_path = Any[], # path of trade-cost changes, t = 1,...,T
    T = Int[],
    max_outer = Int[],
    tol = Float64[],
    damp = Float64[],
    converged = Bool[],
    outer_used = Int[],
    endog_tol = Float64[],                          # tolerance used to judge the endogenous path "converged"
    shock_end_period = Int[],                        # last period p in {0,...,T} where fundamentals still differ from 1
    endog_converged_period = Union{Missing,Int}[],    # first period p* s.t. w_dot/L_dot/u_dot stay within endog_tol
                                                       # of 1 for every period from p* through T; missing if it
                                                       # never happens within the T-period horizon
    endog_convergence_lag = Union{Missing,Int}[],     # endog_converged_period - shock_end_period; missing if the
                                                       # above is missing
    gross_migration_path = Any[],  # Vector of N x M matrices, one per period t=0,...,T: total churn
                                    # (in + out, excluding stayers) through each market (n,j) -- see migration_flows
    net_migration_path = Any[],    # Vector of N x M matrices, one per period t=0,...,T: L_{t+1}-L_t at each
                                    # market (n,j) -- should -> 0 as the labor distribution converges
    result = Any[],          # full named tuple from solve_transition_path_hat -- the estimated
                              # whole transition path (L_path, mu_path, w_path, pi_final,
                              # u_dot_path, omega_dot_path, converged, outer_used)
)

function log_transition_run!(df::DataFrame, label::String, A_shocks, kappa_shocks,
                              w_0, L_0, pi_0, mu_minus1, A_dot_path, kappa_dot_path,
                              T::Int, max_outer::Int, tol::Float64, damp::Float64, result;
                              endog_tol::Float64=1e-3)
    # Last period the fundamentals path itself was still changing (exact == 1.0 is safe: untouched
    # entries are literal `ones(...)`, never arithmetically touched, so no float-tolerance is needed).
    shock_end_period = 0
    for p in 1:length(A_dot_path)
        is_flat = A_dot_path[p] == ones(N,J) && all(kappa_dot_path[p][j] == ones(N,N) for j in 1:J)
        is_flat || (shock_end_period = p)
    end

    # Endogenous "dot" deviation from 1 at period p, using the same "change going into period p"
    # convention as A_dot_path/u_dot_path (w_path[p+1]/w_path[p] = w_p/w_{p-1}, since w_path[s]=w_{s-1}).
    Tu = length(result.u_dot_path)
    dev(p) = max(maximum(abs.(result.w_path[p+1] ./ result.w_path[p] .- 1)),
                 maximum(abs.(result.L_path[p+1] ./ result.L_path[p] .- 1)),
                 maximum(abs.(result.u_dot_path[p] .- 1)))

    # First period after which the endogenous path stays within tolerance through the end of the horizon.
    endog_converged_period = missing
    for p in 1:Tu
        if all(dev(q) < endog_tol for q in p:Tu)
            endog_converged_period = p
            break
        end
    end
    endog_convergence_lag = ismissing(endog_converged_period) ? missing : endog_converged_period - shock_end_period

    gross_migration_path, net_migration_path = migration_flows(result.mu_path, result.L_path)

    push!(df, (
        label=label, A_shocks=A_shocks, kappa_shocks=kappa_shocks,
        w_0=w_0, L_0=L_0, pi_0=pi_0, mu_minus1=mu_minus1,
        A_dot_path=A_dot_path, kappa_dot_path=kappa_dot_path,
        T=T, max_outer=max_outer, tol=tol, damp=damp,
        converged=result.converged, outer_used=result.outer_used,
        endog_tol=endog_tol, shock_end_period=shock_end_period,
        endog_converged_period=endog_converged_period, endog_convergence_lag=endog_convergence_lag,
        gross_migration_path=gross_migration_path, net_migration_path=net_migration_path,
        result=result,
    ))
    return df
end


log_transition_run! (generic function with 1 method)

# Sandbox: Exploring Shock Scenarios

A playground for running `solve_transition_path_hat` under different shocks, without needing to hand-build a `T`-length path of matrices every time. Every scenario below starts the economy at the same baseline -- `(w_temp, L_0, pi_temp)` from the Baseline Levels section above -- and treats its shock as an **unanticipated surprise hitting an economy that was previously at its own steady state**. Concretely, $\mu_{-1}$ is set to `mu_stationary`, the migration shares of the *baseline, unshocked* stationary equilibrium.

This is a deliberate simplification, not the exact Proposition 2 construction: true perfect-foresight $\mu_{-1}$ would need to be derived from the *shocked* economy's own period-0 value function (which requires already knowing the answer -- not something a sandbox meant to explore new scenarios on the fly can do). Getting this exactly right for genuinely unanticipated shocks is what the original paper's Proposition 3 is for, which isn't implemented in this notebook yet.

`run_shock_scenario` takes lists of productivity shocks (`A_shocks`) and/or trade-cost shocks (`kappa_shocks`), each specifying a market, a target multiplicative change (`factor`), and how many periods it phases in over (`ramp` -- `1` for a one-time jump, more for a gradual change), solves the transition, and prints a before/after summary.

In [11]:
# print_labeled: formats an N x M(-ish) matrix with row/column labels for console display
# (originally introduced alongside Notebook 3's 2-period sequential-equilibrium experiment).
# Used below by summarize_transition to report labor distributions and wage changes.
function print_labeled(title, description, M::AbstractMatrix, colnames)
    println(title)
    println("  ", description)
    println("             " * join(rpad.(colnames, 16)))
    for n in 1:size(M,1)
        println("  Region $n:  " * join([rpad(string(round(M[n,c], digits=4)), 16) for c in 1:size(M,2)]))
    end
    println()
end


print_labeled (generic function with 1 method)

In [12]:
# Baseline mu_{-1} for the sandbox: the migration shares of the economy's OWN stationary
# equilibrium under the unshocked baseline wage w_temp (see the markdown above for why this is
# an approximation, not exact Proposition 2 foresight).
U_mkt_stationary = flow_utility_mkt_at(w_temp, A_0, kappa_0)
V_stationary = stationary_V(U_mkt_stationary)
mu_stationary = migration_shares(V_stationary)

function summarize_transition(label::String, result; Tsim::Int=T)
    println("\n", "="^70)
    println(label)
    println("="^70)
    print_labeled(
        "Labor distribution at t=0",
        "mass of households in each region-market, before the shock",
        result.L_path[1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Labor distribution at t=$Tsim (new long-run allocation)",
        "mass of households in each region-market, after the transition",
        result.L_path[Tsim+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Wage change w_$Tsim / w_temp",
        "nominal wage at t=$Tsim relative to the pre-shock baseline",
        result.w_path[Tsim+1] ./ w_temp, ["Sector 1","Sector 2","Sector 3"]
    )
end

# shock spec: (n=region, j=sector, factor=target multiplicative change, ramp=periods to phase in
# over -- 1 for a one-time jump). kappa shock spec adds i=origin region (kappa_dot[j][n,i], the
# cost shipping sector-j goods from i to n, matching the kappa_0 convention used throughout).
function run_shock_scenario(label::String; A_shocks=NamedTuple[], kappa_shocks=NamedTuple[], Tsim::Int=T,
                             max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3)
    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for s in A_shocks
        per_period = s.factor^(1/s.ramp)
        for t in 1:min(s.ramp, Tsim)
            A_dot_path[t][s.n, s.j] *= per_period
        end
    end

    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for s in kappa_shocks
        per_period = s.factor^(1/s.ramp)
        for t in 1:min(s.ramp, Tsim)
            kappa_dot_path[t][s.j][s.n, s.i] *= per_period
        end
    end

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    log_transition_run!(transition_runs, label, A_shocks, kappa_shocks, w_temp, L_0, pi_temp, mu_stationary,
                         A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp, result; endog_tol=endog_tol)
    summarize_transition(label, result; Tsim=Tsim)
    return result
end


# Alternative to run_shock_scenario for i.i.d. RANDOM (rather than deterministic ramped) shocks:
# every period in `periods` draws an independent log-normal multiplicative shock (mean 1 in log,
# so E[log(dot)]=0) for every region-sector productivity and every off-diagonal trade cost;
# periods outside that range are left at dot=1 (no change), same "shock active, then frozen"
# structure as run_shock_scenario's ramped shocks. `seed`, if given, reseeds the global RNG first
# so the draw is reproducible across re-runs.
function run_random_shock_scenario(label::String; periods=1:50, sigma_A::Float64=0.05, sigma_kappa::Float64=0.05,
                                    Tsim::Int=T, seed::Union{Int,Nothing}=nothing,
                                    max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3)
    seed === nothing || Random.seed!(seed)

    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        A_dot_path[p] = exp.(sigma_A .* randn(N,J))
    end

    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        for j in 1:J, n in 1:N, i in 1:N
            n == i && continue
            kappa_dot_path[p][j][n,i] = exp(sigma_kappa * randn())
        end
    end

    shock_spec = (kind=:iid_random_walk, sigma_A=sigma_A, sigma_kappa=sigma_kappa, periods=periods, seed=seed)

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    log_transition_run!(transition_runs, label, [shock_spec], [shock_spec], w_temp, L_0, pi_temp, mu_stationary,
                         A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp, result; endog_tol=endog_tol)
    summarize_transition(label, result; Tsim=Tsim)
    return result
end


run_random_shock_scenario (generic function with 1 method)

In [13]:
# Scenario 1: no shock at all. Since L_0 isn't itself the stationary distribution implied by
# mu_stationary, this still shows real dynamics -- pure convergence toward the baseline's own
# steady state -- a useful reference point for judging how much of the change in the shocked
# scenarios below is "just settling down" versus actually caused by the shock.
scenario_none = run_shock_scenario("Scenario 1: No shock (pure convergence to baseline steady state)")

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.808641567739414e-9)

Scenario 1: No shock (pure convergence to baseline steady state)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0278          0.1555          0.3684          0.4483          
  Region 2:  3.0278          0.4862          0.3403          0.1458          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.686191223606516 0.22270326478503655 0.49616374488216036 0.5958656020149583; 2.686191223606516 0.6428735233636291 0.4612885794337561 0.20872283830742924], [2.9662416664911198 0.16377223697566004 0.3920212818295205 0.47785385626323823; 2.9662416664911198 0.5186223639862972 0.36176011214668985 0.15348681581635615], [3.016467509061154 0.15684066376086198 0.3727672887989706 0.4538461440829731; 3.016467509061154 0.49238644950793337 0.3441670572788329 0.1470573784481225], [3.0256702471806425 0.1557510860772527 0.36922986715896416 0.44929978922970304; 3.0256702471806425 0.48735164103539175 0.3409947112957242 0.1460324108416822], [3.027366261735465 0.15555938303913208 0.36857808945810144 0.44845444398585477; 3.027366261735465 0.48641065813058904 0.34041455292606465 0.14585034898932936], [3.0276793289128605 0.1555244700205106 0.36845779665649997 0.44829802767081606; 3.0276793289128605 0.4862362015156075 0.3403077910024702 0.1458170553083775], [3.

In [14]:
# Scenario 2: one-time productivity jump, region 1 sector 1, +20%.
scenario_up = run_shock_scenario("Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=1)])

solve_transition_path_hat converged after 20 outer iterations (max u_dot change = 8.12633360602888e-9)

Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0174          0.174           0.3703          0.4506          
  Region 2:  3.0174          0.4817          0.342           0.1465          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6749477818462064 0.24719418900980827 0.497584811703744 0.5975642719445258; 2.6749477818462064 0.6358159149114195 0.4626125957446996 0.2093326529933899], [2.9556253370850585 0.18340715441973254 0.39389001596268647 0.48012941705730205; 2.9556253370850585 0.5136184116661995 0.36348519240046223 0.1542191343234998], [3.006072168012991 0.17551131922260238 0.374658333098933 0.4561551834873423; 3.006072168012991 0.48783053151787004 0.345910045111709 0.14779025153556027], [3.0153231584732243 0.17425632728625545 0.3711184950263351 0.4516060755444784; 3.0153231584732243 0.48287312735147414 0.3427353473396824 0.14676431050532482], [3.0170292207937424 0.17403533676618274 0.37046557177405776 0.45075921800599117; 3.0170292207937424 0.48194526265124393 0.34215417706177514 0.1465819921532634], [3.017344359520465 0.17399509326149415 0.3703449697182774 0.4506023858066896; 3.017344359520465 0.4817730508142469 0.34204714692869453 0.14654863442966637], [3.01

In [15]:
# Scenario 3: one-time productivity DECLINE, region 1 sector 1, -20% -- the mirror image of
# Scenario 2, in the spirit of a "China shock"-style competitiveness hit to a single market.
scenario_down = run_shock_scenario("Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)";
    A_shocks = [(n=1, j=1, factor=0.8, ramp=1)])

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 9.840530612592602e-9)

Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0387          0.1353          0.3667          0.4462          
  Region 2:  3.0387          0.4906          0.3387          0.1451          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Regi

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.698253459336259 0.19550816002359253 0.49500803502782587 0.5944850375380659; 2.698253459336259 0.6500552258452947 0.4602114125232383 0.2082252103694653], [2.977462595873722 0.14234856848087613 0.3903274655091282 0.47579475222220613; 2.977462595873722 0.5235917627086548 0.3601951305446792 0.15281712878701087], [3.027450419184213 0.13645913339956442 0.3710397331058076 0.45173876351299425; 3.027450419184213 0.49690315362900755 0.3425739090855879 0.14638446889861156], [3.03660525763501 0.1355420589696896 0.36750225217999444 0.4471918410431931; 3.03660525763501 0.49179160271365224 0.339401750201937 0.1453599796215126], [3.038291768178292 0.13538057387911379 0.3668510133613736 0.4463471982779784; 3.038291768178292 0.49083756561008546 0.33882207419717636 0.14517803831768705], [3.0386029417008324 0.13535114104085932 0.3667308885914738 0.44619101018520796; 3.0386029417008324 0.49066084424219253 0.33871545713421813 0.14514477540438264], [3.0386603

In [16]:
# Scenario 4: the SAME +20% productivity shock as Scenario 2, but phased in gradually over 5
# periods instead of hitting all at once -- compare the transition speed/shape to Scenario 2's
# new long-run allocation (should end up close to the same place, just arriving more slowly).
scenario_ramp = run_shock_scenario("Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=5)])

solve_transition_path_hat converged after 21 outer iterations (max u_dot change = 6.080586256729248e-9)

Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0161          0.1748          0.3708          0.4512          
  Region 2:  3.0161          0.4819          0.3425          0.1467          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6837134725907803 0.22766959680525076 0.49652554650056396 0.5963028599039891; 2.6837134725907803 0.6415810938262441 0.46162376953425915 0.20887018824813397], [2.9618264282147915 0.171535712672075 0.3928364126646868 0.47885327583236853; 2.9618264282147915 0.516815888769519 0.3625098632793404 0.15379599035242741], [3.00979000813395 0.16817157634939672 0.37406237186710883 0.4554355092247824; 3.00979000813395 0.4898456321471193 0.34535758929235866 0.14754730485133408], [3.016603087248206 0.17096236160274864 0.3710245621898469 0.45150229784519413; 3.016603087248206 0.483948685272449 0.34264452742161616 0.14671139117173443], [3.016080077289137 0.17461433261483608 0.3708072768943768 0.4511888827656715; 3.016080077289137 0.48206928227281576 0.3424642971530589 0.14669577372096837], [3.0160657323159508 0.17478290520960033 0.3708038691136056 0.4511788711302012; 3.0160657323159508 0.48193695632225797 0.34246381614861476 0.14670211744382086], [3.0160

In [17]:
# Scenario 5: SYMMETRIC shock -- both regions get the same +20% boost in sector 1, so relative
# competitiveness (and hence trade shares and migration incentives, which only respond to
# RELATIVE productivity) shouldn't move much, unlike the one-sided Scenario 2.
scenario_symmetric = run_shock_scenario("Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=1), (n=2, j=1, factor=1.2, ramp=1)])

solve_transition_path_hat converged after 24 outer iterations (max u_dot change = 7.960827330677489e-9)

Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.9744          0.1637          0.3887          0.4731          
  Region 2:  2.9744          0.5132          0.3589          0.1535          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Regi

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.634138522861165 0.23161376018480395 0.5158166007634007 0.6194031722187383; 2.634138522861165 0.6682357069360647 0.47958023796194693 0.21707347621271456], [2.911687815086448 0.17219891552996822 0.4127548083802765 0.5032371885151034; 2.911687815086448 0.5462086161298534 0.3808456048120472 0.1613792364598541], [2.962647628385161 0.16512631839746733 0.39321804411318284 0.478912015147753; 2.962647628385161 0.5196456093809051 0.3629774018686724 0.1548253543216944], [2.9722127371677938 0.16399664419535595 0.3895401370578268 0.4741845221234615; 2.9722127371677938 0.5144109263102868 0.35967893045288424 0.15376336552459502], [2.974019021454343 0.1637936734473093 0.3888457451202314 0.4732832584501105; 2.974019021454343 0.5134075513890166 0.3590611067880876 0.15357062189655654], [2.9743607002510006 0.1637558383983087 0.3887144140630717 0.47311233131149283; 2.9743607002510006 0.513216856930506 0.35894462507715086 0.15353453371746562], [2.97442536396

In [18]:
# Scenario 6: trade liberalization -- the iceberg cost shipping sector-1 goods from region 2 to
# region 1 falls by 30% (kappa_dot = 0.7), with no productivity change at all -- a pure trade-cost
# shock, exercising the kappa_shocks side of the wrapper.
scenario_trade = run_shock_scenario("Scenario 6: 30% reduction in trade costs, region 2 -> region 1, sector 1";
    kappa_shocks = [(n=1, j=1, i=2, factor=0.7, ramp=1)])

solve_transition_path_hat converged after 24 outer iterations (max u_dot change = 6.216107406586957e-9)

Scenario 6: 30% reduction in trade costs, region 2 -> region 1, sector 1
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.9822          0.1607          0.4057          0.4869          
  Region 2:  2.9822          0.4988          0.3399          0.1437          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.642436165015354 0.22756541278679 0.5370302147113458 0.637175592419738; 2.642436165015354 0.6520590257076302 0.4569884485528831 0.20430897579090523], [2.9196009296186376 0.1690209121886385 0.43111112323759854 0.5182423240281789; 2.9196009296186376 0.5309612319363908 0.36052018585788137 0.1509423635140368], [2.970426367905451 0.16207296687541445 0.41051134563500036 0.49299934765004; 2.970426367905451 0.5050627063202673 0.34363281776344695 0.1448680799449289], [2.9799618544885442 0.16095752777578884 0.40659143021482635 0.4880733282984114; 2.9799618544885442 0.4999977531709189 0.34056216302939873 0.14389408853356653], [2.981761788242732 0.16075636261669737 0.40584817755393077 0.48713310260612863; 2.981761788242732 0.4990302260643416 0.3399907050440985 0.14371784962933798], [2.9821021122768174 0.16071880719397832 0.40570739608147155 0.48695476207697497; 2.9821021122768174 0.4988466481894958 0.3398832694881664 0.1436848924162782], [2.98216648

In [19]:
print(transition_runs)

6×22 DataFrame
 Row │ label                              A_shocks                           kappa_shocks                       w_0                                L_0                                pi_0                               mu_minus1                          A_dot_path                         kappa_dot_path                     T      max_outer  tol      damp     converged  outer_used  endog_tol  shock_end_period  endog_converged_period  endog_convergence_lag  gross_migration_path               net_migration_path                 result                            
     │ String                             Any                                Any                                Any                                Any                                Any                                Any                                Any                                Any                                Int64  Int64      Float64  Float64  Bool       Int64       Float64    Int64             Union{Missing

In [20]:
# Scenario 7: "everything" shock -- every region-sector productivity up 20% and every off-diagonal
# trade cost down 30% (self-shipping cost kappa[n,n] is left alone, as always), all phased in
# gradually over 50 periods (ramp=50) instead of the 1- or 5-period ramps used above, simulated
# out to Tsim=200 -- 150 periods of buffer AFTER the shock finishes ramping, so the
# fundamentals-to-endogenous convergence lag is actually observable (Tsim=50 left no room past
# shock_end_period=50 to see the endogenous path settle).
A_shocks_everything = vec([(n=n, j=j, factor=1.2, ramp=50) for n in 1:N, j in 1:J])
kappa_shocks_everything = [(n=n, j=j, i=i, factor=0.7, ramp=50) for n in 1:N, j in 1:J, i in 1:N if n != i]

scenario_everything = run_shock_scenario("Scenario 7: all fundamentals shocked, ramped over 50 periods";
    A_shocks = A_shocks_everything, kappa_shocks = kappa_shocks_everything, Tsim = 200)


solve_transition_path_hat converged after 26 outer iterations (max u_dot change = 9.24135745705712e-9)

Scenario 7: all fundamentals shocked, ramped over 50 periods
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.5579          0.2475          0.5401          0.655           
  Region 2:  2.5579          0.7102          0.5012          0.2301          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.679033433800976 0.2239009663063003 0.49889850838381156 0.5991563786245071; 2.679033433800976 0.6463983477672601 0.46376926576532274 0.20980966555084463], [2.9525661630631115 0.16589219379803818 0.3972581948370695 0.4842648164092123; 2.9525661630631115 0.5255465849198239 0.3664871027745842 0.15541878113504828], [2.995835690105074 0.16005067264935124 0.3806565246404253 0.4635085538936447; 2.995835690105074 0.5028305429392881 0.351294280448988 0.14998804521815473], [2.997712608589907 0.16012137360575227 0.3799092674992391 0.46237822740399886; 2.997712608589907 0.5014916016371316 0.3506499958635761 0.1500243168104897], [2.9918776541981233 0.16113033946286387 0.3821214820653941 0.4650397974600743; 2.9918776541981233 0.5043459529864418 0.3526667508013076 0.15094036882767267], [2.984533548502104 0.1623278330932644 0.3849080236472212 0.4684413162638561; 2.984533548502104 0.5080226625155971 0.35519867257539517 0.1520343949004605], [2.97683525654

In [21]:
# Scenario 8: same "everything" scope as Scenario 7, but instead of a smooth deterministic ramp,
# every region-sector productivity and every off-diagonal trade cost gets its own independent
# random multiplicative shock each period for periods 1-50 (fixed at dot=1 -- no further change --
# from period 51 onward), simulated out to Tsim=200 for the same 150-period convergence buffer as
# Scenario 7. Tests whether random period-to-period fundamentals (as opposed to a smooth ramp)
# change how long the endogenous path takes to settle once the fundamentals stop moving.
scenario_random = run_random_shock_scenario("Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50";
    periods = 1:50, sigma_A = 0.05, sigma_kappa = 0.05, Tsim = 200, seed = 2)


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 8.434787757138906e-9)

Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6102          0.2003          0.5429          0.6257          
  Region 2:  2.6102          0.7627          0.4702          0.1778          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.682657717418846 0.21062674199206097 0.4930479714621486 0.590672714115908; 2.682657717418846 0.6504691435250411 0.46690450683444884 0.222963487232699], [2.98216967723223 0.146644175454939 0.39810900788799336 0.4633809721543658; 2.98216967723223 0.5176124107256724 0.3429338964365744 0.16698018287599234], [3.0067663880718833 0.1506254799628298 0.3892355392705267 0.44711208881000325; 3.0067663880718833 0.5034567583725185 0.33099900607976424 0.16503835136059042], [3.019475245316762 0.13442100597245696 0.39010109912204277 0.44170428670667816; 3.019475245316762 0.5018752877670238 0.3231241462320975 0.16982368356617766], [3.005761479983189 0.13162081309086981 0.4004673519167107 0.44762658140340383; 3.005761479983189 0.5120375845243724 0.32293939063417293 0.17378531846409273], [2.987167218700723 0.1308672491441078 0.4155977779574943 0.45701573434786535; 2.987167218700723 0.5245995491520029 0.3210496369985775 0.17653561499850678], [2.954311601998

# Sandbox: Parameter Sensitivity Scenarios

Scenarios 9-11 below probe how the fundamentals-to-endogenous convergence lag (see the Logging
section) responds to (a) mobility frictions and (b) economy size, all using the **same shock
process as Scenario 8**: i.i.d. log-normal productivity/trade-cost shocks, `sigma_A = sigma_kappa
= 0.05`, active over periods 1-50, `seed = 2`, simulated out to `Tsim = 200` (the same 150-period
post-shock buffer).

**N=10 region scenarios are deferred.** Bumping `N` to 10 (alone or combined with `J=15`) makes
`solve_temporary_equilibrium`/`solve_temp_eq_hat`'s feasibility formulation (`Min 0` subject to
the market-clearing equalities) fail outright in Ipopt -- confirmed `LOCALLY_INFEASIBLE` with
genuine ~10-20% constraint violations, not numerical noise. A least-squares reformulation
(`Min sum(residual^2)`, same economics) fixes this, but changing the solver was explicitly
deferred, so those scenarios aren't run here.

Scenario 9 keeps `N=2, J=3` (Scenarios 1-8's baseline `w_temp`/`pi_temp`/`L_0`/`A_0`/`kappa_0`
untouched) and only overrides the *off-diagonal* mobility cost `tau_mig` -- the cost of staying
in the same market remains 0, per the paper's convention. Scenario 10 mirrors it with a
near-infinite mobility cost. Scenario 11 regenerates the fundamentals at `J=15` (`N` unchanged at
2); because that requires reassigning the shared globals (`N`, `J`, `M`, `A_0`, `kappa_0`, ...),
it's run **last** and permanently leaves the notebook's state at `N=2, J=15` -- re-run the
Parameters/Baseline Levels cells above if you want to get back to the `J=3` baseline afterward.

In [22]:
# run_taumig_variation_scenario: reruns the transition under a modified off-diagonal mobility
# cost (0 = free mobility, a large value = effectively no mobility), holding the production side
# (w_temp, pi_temp, L_0, A_0, kappa_0) fixed at the Scenario 1-8 baseline -- tau_mig only enters
# the model through mu_stationary (stationary_V/migration_shares), never through
# solve_temporary_equilibrium/solve_temp_eq_hat, so nothing else needs to be re-solved. The global
# tau_mig is temporarily overridden and restored afterward so later cells are unaffected.
function run_taumig_variation_scenario(label::String, cost_value::Float64;
                                        periods=1:50, sigma_A::Float64=0.05, sigma_kappa::Float64=0.05,
                                        Tsim::Int=T, seed::Int=2, max_outer::Int=100, tol::Float64=1e-8,
                                        damp::Float64=0.5, endog_tol::Float64=1e-3)
    global tau_mig
    tau_mig_original = tau_mig
    tau_mig = [(n == i && j == k) ? 0.0 : cost_value for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

    U_mkt_v = flow_utility_mkt_at(w_temp, A_0, kappa_0)
    V_v = stationary_V(U_mkt_v)
    mu_v = migration_shares(V_v)

    # Same i.i.d. shock-generation process as run_random_shock_scenario (Scenario 8), so with the
    # same N, J and seed this reproduces the identical shock realization as Scenario 8 -- isolating
    # tau_mig as the only thing that differs.
    Random.seed!(seed)
    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        A_dot_path[p] = exp.(sigma_A .* randn(N,J))
    end
    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        for j in 1:J, n in 1:N, i in 1:N
            n == i && continue
            kappa_dot_path[p][j][n,i] = exp(sigma_kappa * randn())
        end
    end
    shock_spec = (kind=:iid_random_walk, sigma_A=sigma_A, sigma_kappa=sigma_kappa, periods=periods, seed=seed)

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_v, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    log_transition_run!(transition_runs, label, [shock_spec], [shock_spec], w_temp, L_0, pi_temp, mu_v,
                         A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp, result; endog_tol=endog_tol)
    summarize_transition(label, result; Tsim=Tsim)

    tau_mig = tau_mig_original
    return result
end


run_taumig_variation_scenario (generic function with 1 method)

In [23]:
# Scenario 9: migration costs set to 0 everywhere (off-diagonal) -- frictionless mobility.
# Same shock realization as Scenario 8 (same N, J, seed=2), so any difference in the convergence
# lag vs. Scenario 8 is attributable to mobility frictions alone.
scenario_taumig_zero = run_taumig_variation_scenario(
    "Scenario 9: migration costs = 0 everywhere (frictionless mobility)", 0.0; Tsim=200)


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 5.557526883137598e-9)

Scenario 9: migration costs = 0 everywhere (frictionless mobility)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.0911          0.3169          0.7396          0.8349          
  Region 2:  2.0911          0.9865          0.6582          0.2818          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Re

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.41454978015264 0.26468442363639694 0.5911425975251449 0.7020118224489921; 2.41454978015264 0.767370650827458 0.5663163791159148 0.2793745661408131], [2.4465806949231057 0.25114813432700694 0.6037119402117792 0.685425267481895; 2.4465806949231057 0.75486867326646 0.5288434421940281 0.28284115267261967], [2.406342300748488 0.27000953698907004 0.6182559735013186 0.6968394421930754; 2.406342300748488 0.7724271065019059 0.5390386112740606 0.2907447280435945], [2.4215638757028333 0.2417791830348236 0.6211895207963437 0.690989236113272; 2.4215638757028333 0.7739190916769316 0.5291491456945872 0.299846071278378], [2.4047483751133667 0.23719993128268593 0.6345081642725264 0.698182448203837; 2.4047483751133667 0.7866297566191994 0.5286035044832499 0.3053794449117697], [2.390908206487598 0.23434724223025438 0.6513395358547122 0.7058136805498294; 2.390908206487598 0.7975143991122431 0.5216182162417501 0.30755051303601727], [2.3654333016667892 0.240

In [24]:
# Scenario 10: migration costs set to 100000 everywhere (off-diagonal) -- effectively no mobility
# (moving is prohibitively costly, so households are pinned to their initial market). Same shock
# realization as Scenario 8 and Scenario 9.
scenario_taumig_high = run_taumig_variation_scenario(
    "Scenario 10: migration costs = 100000 everywhere (effectively no mobility)", 100000.0; Tsim=200)


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 6.2722032012629825e-9)

Scenario 10: migration costs = 100000 everywhere (effectively no mobility)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3    

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]  …  [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]], mu_path = [[1.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 1.0 0.0 0.0 0.0;;;; 0.0 1.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0;;;; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 0.0 1.0 0.0;;;; 0.0 0.0 0.0 1.0; 0.0 0.0 0.0 0.0;;

In [25]:
# run_sector_scaling_scenario: rebuilds the fundamentals at a new sector count J_new (N unchanged),
# re-solves the temporary-equilibrium baseline and mu_stationary from scratch (mirroring the
# Parameters and Baseline Levels cells above), then runs the same i.i.d. shock process as
# Scenario 8. Unlike run_taumig_variation_scenario, this permanently reassigns the shared globals
# (N, J, M, A_0, kappa_0, ...) since the production side itself is changing size -- that's why it's
# run last, after the tau_mig scenarios have already used the original J=3 baseline.
#
# NOTE: this does NOT reuse the shared `summarize_transition` helper -- that function hardcodes
# 3-sector column labels and reads the *global* w_temp (still the J=3 baseline) rather than this
# scenario's own w_temp_v, so at J_new != 3 it throws a DimensionMismatch. The block below is a
# J_new-aware equivalent that uses w_temp_v and generates the right number of sector labels.
function run_sector_scaling_scenario(label::String, J_new::Int;
                                      seed_fundamentals::Int=1, periods=1:50, sigma_A::Float64=0.05,
                                      sigma_kappa::Float64=0.05, Tsim::Int=T, seed_shock::Int=2,
                                      max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5,
                                      endog_tol::Float64=1e-3)
    global N, J, M, L_0, A_0, B, w_0, b, kappa_0, theta, eta, gamma, tau_mig, alpha
    J = J_new
    M = J + 1

    Random.seed!(seed_fundamentals)
    L_0 = ones(N, M)
    A_0 = rand(N, J)
    B = ones(N, J)
    w_0 = ones(N, J)
    b = ones(N)
    kappa_0 = [ones(N,N) for _ in 1:J]
    theta = fill(4.0, J)
    eta = fill(2.0, N, J)
    gamma = ones(N, J)
    tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
    alpha = rand(J); alpha = alpha ./ sum(alpha)

    w_temp_v, pi_temp_v, X_temp_v, status_temp_v = solve_temporary_equilibrium(L_0)
    println("  [J=$J_new] baseline temporary-equilibrium solver status: ", status_temp_v)

    U_mkt_v = flow_utility_mkt_at(w_temp_v, A_0, kappa_0)
    V_v = stationary_V(U_mkt_v)
    mu_v = migration_shares(V_v)

    Random.seed!(seed_shock)
    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        A_dot_path[p] = exp.(sigma_A .* randn(N,J))
    end
    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        for j in 1:J, n in 1:N, i in 1:N
            n == i && continue
            kappa_dot_path[p][j][n,i] = exp(sigma_kappa * randn())
        end
    end
    shock_spec = (kind=:iid_random_walk, sigma_A=sigma_A, sigma_kappa=sigma_kappa, periods=periods, seed=seed_shock)

    result = solve_transition_path_hat(w_temp_v, L_0, pi_temp_v, mu_v, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    log_transition_run!(transition_runs, label, [shock_spec], [shock_spec], w_temp_v, L_0, pi_temp_v, mu_v,
                         A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp, result; endog_tol=endog_tol)

    println("\n", "="^70)
    println(label)
    println("="^70)
    labor_colnames = vcat(["Non-employment"], ["Sector $j" for j in 1:J])
    print_labeled("Labor distribution at t=0", "mass of households in each region-market, before the shock",
                  result.L_path[1], labor_colnames)
    print_labeled("Labor distribution at t=$Tsim (new long-run allocation)",
                  "mass of households in each region-market, after the transition",
                  result.L_path[Tsim+1], labor_colnames)
    wage_colnames = ["Sector $j" for j in 1:J]
    print_labeled("Wage change w_$Tsim / w_temp", "nominal wage at t=$Tsim relative to the pre-shock baseline",
                  result.w_path[Tsim+1] ./ w_temp_v, wage_colnames)

    return result
end


run_sector_scaling_scenario (generic function with 1 method)

In [26]:
# Scenario 11: J=15 sectors (N=2 unchanged). Regenerates A_0, kappa_0, etc. at the new size (seed=1,
# same convention as the original Parameters cell) and re-solves the baseline from scratch, then
# applies the same i.i.d. shock process as Scenario 8 (seed=2). NOTE: this permanently leaves the
# notebook's global state at N=2, J=15 -- re-run the Parameters/Baseline Levels cells above to
# return to the J=3 baseline.
scenario_sectors15 = run_sector_scaling_scenario(
    "Scenario 11: J=15 sectors (N=2)", 15; Tsim=200)


  [J=15] baseline temporary-equilibrium solver status: LOCALLY_SOLVED
solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 6.803137608812904e-9)

Scenario 11: J=15 sectors (N=2)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        Sector 4        Sector 5        Sector 6        Sector 7        Sector 8        Sector 9        Sector 10       Sector 11       Sector 12       Sector 13       Sector 14       Sector 15       
  Region 1:  1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0      

(L_path = [[1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0], [10.305755768941703 0.17296337283030333 … 0.404076924775912 0.12048902718542369; 10.305755768941703 0.3160559321074949 … 0.6671555228752206 0.2002029800762689], [11.504738908900036 0.12484108985867737 … 0.3182531448379036 0.08679921775543321; 11.504738908900036 0.253934420942132 … 0.5311241608126288 0.1608844676080147], [11.65098871067111 0.1316105990310174 … 0.32152539889465315 0.08710482732779051; 11.65098871067111 0.23973807871747477 … 0.5034056520053666 0.15387055752903672], [11.65640666122915 0.13427444884011638 … 0.32120163258896933 0.08868499350143998; 11.65640666122915 0.23789795890028917 … 0.5025953922414719 0.15275600303614467], [11.62483012663942 0.13483898703195768 … 0.3221826477548319 0.0899802551070532; 11.62483012663942 0.23968270829961752 … 0.5067209301436335 0.15337791858287736], [11.634471115467925 0.12738863573556825 … 0.33266969513032146 0.0880018116894115; 11.634471115467925 0.2424433768057719 … 0.4970557862485815 

In [27]:
# Convergence-lag summary for Scenarios 8-11 (same shock process throughout; only tau_mig or J
# differs), pulled straight from the transition_runs log.
summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
println(transition_runs[in(["Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50",
                             "Scenario 9: migration costs = 0 everywhere (frictionless mobility)",
                             "Scenario 10: migration costs = 100000 everywhere (effectively no mobility)",
                             "Scenario 11: J=15 sectors (N=2)"]).(transition_runs.label), summary_cols])


4×4 DataFrame
 Row │ label                              shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Scenario 8: all fundamentals, i.…                50                      52                      2
   2 │ Scenario 9: migration costs = 0 …                50                      51                      1
   3 │ Scenario 10: migration costs = 1…                50                      51                      1
   4 │ Scenario 11: J=15 sectors (N=2)                  50                      52                      2


In [28]:
# Migration flow diagnostics: for every scenario, show the max-abs NET flow (should shrink toward
# 0 as the labor distribution settles) and max GROSS flow (should stay bounded away from 0 for any
# finite tau_mig -- idiosyncratic taste shocks keep people reshuffling even at the new steady
# state -- and should be ~0 throughout only in the no-mobility scenario) at the start, middle, and
# final period of each scenario's horizon.
println("\n==== MIGRATION FLOW DIAGNOSTICS (max over all markets n,j) ====")
for row in eachrow(transition_runs)
    gp, np = row.gross_migration_path, row.net_migration_path
    Tlen = length(gp)
    mid = max(1, Tlen ÷ 2)
    println(row.label)
    println("  net  max|.|  -- t=1: ", round(maximum(abs.(np[1])), digits=6),
            "   t=$mid: ", round(maximum(abs.(np[mid])), digits=6),
            "   t=$Tlen (final): ", round(maximum(abs.(np[end])), digits=8))
    println("  gross max    -- t=1: ", round(maximum(gp[1]), digits=6),
            "   t=$mid: ", round(maximum(gp[mid]), digits=6),
            "   t=$Tlen (final): ", round(maximum(gp[end]), digits=8))
    println()
end



==== MIGRATION FLOW DIAGNOSTICS (max over all markets n,j) ====
Scenario 1: No shock (pure convergence to baseline steady state)
  net  max|.|  -- t=1: 1.686191   t=5: 0.001696   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.514671   t=5: 2.509507   t=11 (final): 2.50953249

Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)
  net  max|.|  -- t=1: 1.674948   t=5: 0.001706   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.506435   t=5: 2.509934   t=11 (final): 2.50996636

Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)
  net  max|.|  -- t=1: 1.698253   t=5: 0.001687   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.523526   t=5: 2.508898   t=11 (final): 2.50891689

Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)
  net  max|.|  -- t=1: 1.683713   t=5: 0.003652   t=11 (final): 0.0
  gross max    -- t=1: 2.512856   t=5: 2.50989   t=11 (final): 2.50995726

Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)
  net  max|.

# Simplified Scenarios

Sandbox for working through simple, hand-picked scenarios one at a time (as opposed to the
batch sweeps above). Cells here will be added and run per scenario as requested.

In [34]:
# This section runs after Scenario 11 ("J=15 sectors"), which PERMANENTLY reassigns the
# shared globals (N, J, M, L_0, A_0, kappa_0, tau_mig, ...) to a 15-sector economy and never
# restores them (see that cell's own note). So this section is self-contained: it resets
# everything back to the N=2, J=3 baseline fundamentals (identical to the original Parameters
# cell, same seed=1 draw order) and re-solves w_temp/pi_temp/mu_stationary from scratch, every
# time it's run -- regardless of what ran above it.
global N, J, M, L_0, A_0, B, w_0, b, kappa_0, theta, eta, gamma, beta, tau_mig, alpha, nu, T

N = 2
J = 3
M = J + 1

L_0 = ones(N,M)

Random.seed!(1)
A_0 = rand(N,J)

B = ones(N,J)
w_0 = ones(N,J)
b = ones(N)
kappa_0 = [ones(N,N) for _ in 1:J]

theta = fill(4.0, J)
eta = fill(2.0, N, J)
gamma = ones(N, J)
beta = 0.95
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J)
alpha = alpha ./ sum(alpha)

nu = 1.0
T = 10

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)
println("[Simplified Scenarios reset] baseline temporary-equilibrium solver status: ", status_temp)

U_mkt_stationary = flow_utility_mkt_at(w_temp, A_0, kappa_0)
V_stationary = stationary_V(U_mkt_stationary)
mu_stationary = migration_shares(V_stationary);

[Simplified Scenarios reset] baseline temporary-equilibrium solver status: LOCALLY_SOLVED


In [35]:
# Separate log for this section: mirrors transition_runs' schema exactly (via log_transition_run!)
# but keeps these one-off, hand-picked scenarios out of the batch-sweep log above.
simplified_scenario_runs = DataFrame(
    label = String[],
    A_shocks = Any[],
    kappa_shocks = Any[],
    w_0 = Any[],
    L_0 = Any[],
    pi_0 = Any[],
    mu_minus1 = Any[],
    A_dot_path = Any[],
    kappa_dot_path = Any[],
    T = Int[],
    max_outer = Int[],
    tol = Float64[],
    damp = Float64[],
    converged = Bool[],
    outer_used = Int[],
    endog_tol = Float64[],
    shock_end_period = Int[],
    endog_converged_period = Union{Missing,Int}[],
    endog_convergence_lag = Union{Missing,Int}[],
    gross_migration_path = Any[],
    net_migration_path = Any[],
    result = Any[],
)

# Unlike summarize_transition (which only prints t=0 and t=Tsim), this section only cares about
# t=1,...,3, so print every period in that window: labor distribution, wage relative to the
# pre-shock baseline, and gross/net migration flow into each market.
function summarize_simple_scenario(label::String, result, gross_migration_path, net_migration_path;
                                    t_start::Int=1, t_end::Int=3)
    println("\n", "="^70)
    println(label)
    println("="^70)
    for t in t_start:t_end
        println("-- t=$t --")
        print_labeled(
            "Labor distribution",
            "mass of households in each region-market at t=$t",
            result.L_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Wage change w_$t / w_temp",
            "nominal wage at t=$t relative to the pre-shock baseline",
            result.w_path[t+1] ./ w_temp, ["Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Net migration flow (L_$t - L_$(t-1))",
            "should -> 0 as the labor distribution converges",
            net_migration_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Gross migration flow",
            "total churn (in+out, excl. stayers) into each market -- stays bounded away from 0",
            gross_migration_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
    end
end

# Same shock-spec convention as run_shock_scenario, but logs into simplified_scenario_runs and
# defaults to a Tsim=3 horizon (per period t=1,...,3) with a period-by-period summary instead of
# summarize_transition's start/end snapshot.
function run_simple_scenario(label::String; A_shocks=NamedTuple[], kappa_shocks=NamedTuple[], Tsim::Int=3,
                              max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3,
                              t_start::Int=1, t_end::Int=3)
    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for s in A_shocks
        per_period = s.factor^(1/s.ramp)
        for t in 1:min(s.ramp, Tsim)
            A_dot_path[t][s.n, s.j] *= per_period
        end
    end

    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for s in kappa_shocks
        per_period = s.factor^(1/s.ramp)
        for t in 1:min(s.ramp, Tsim)
            kappa_dot_path[t][s.j][s.n, s.i] *= per_period
        end
    end

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    log_transition_run!(simplified_scenario_runs, label, A_shocks, kappa_shocks, w_temp, L_0, pi_temp, mu_stationary,
                         A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp, result; endog_tol=endog_tol)
    gross_migration_path, net_migration_path = migration_flows(result.mu_path, result.L_path)
    summarize_simple_scenario(label, result, gross_migration_path, net_migration_path; t_start=t_start, t_end=t_end)
    return result
end

run_simple_scenario (generic function with 1 method)

In [36]:
# Baseline scenario: current fundamentals as-is (N=2 regions, J=3 sectors), no A or kappa
# shocks -- just L_0 relaxing toward the stationary allocation implied by mu_stationary under
# w_temp, over t=1,...,5.
run_simple_scenario("Baseline: N=2, J=3, no shock");

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.798724167505043e-9)

Baseline: N=2, J=3, no shock
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6862          0.2227          0.4962          0.5959          
  Region 2:  2.6862          0.6429          0.4613          0.2087          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.94            0.8722          
  Region 2:  0.8089          0.9538          1.0758          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2801          -0.0589         -0.1041         -0.118          
  Region 2:  0.2801          -0.1243         -

In [37]:
# Scenario: region 1, sector 1 productivity jumps +50% in period 1 (one-time, ramp=1),
# nothing else shocked.
run_simple_scenario("Region 1, Sector 1 productivity +50%";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=1)]);

solve_transition_path_hat converged after 23 outer iterations (max u_dot change = 5.217704046600602e-9)

Region 1, Sector 1 productivity +50%
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6593          0.28            0.5001          0.6005          
  Region 2:  2.6593          0.6255          0.4649          0.2104          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.7433          0.6897          
  Region 2:  0.6156          0.7542          0.8507          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2813          -0.0698         -0.1031         -0.1167         
  Region 2:  0.2813          -0.1194  

In [38]:
# Scenario: iceberg trade cost between region 1 and 2 doubles, both directions, every sector,
# one-time jump in period 1. No productivity shock.
run_simple_scenario("Region 1 <-> 2 trade costs double";
    kappa_shocks=[(n=n_, j=j, i=i_, factor=2.0, ramp=1) for (n_,i_) in [(1,2),(2,1)] for j in 1:J]);

solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 5.013182757807044e-9)

Region 1 <-> 2 trade costs double
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.8478          0.2429          0.4138          0.488           
  Region 2:  2.8478          0.5269          0.4039          0.2288          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.7062          0.642           
  Region 2:  0.5975          0.7574          1.0853          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2865          -0.068          -0.1024         -0.1156         
  Region 2:  0.2865          -0.1226     

In [39]:
# Scenario: migration cost = 0 everywhere (frictionless mobility), no productivity/trade-cost
# shock. tau_mig only enters through mu_stationary (stationary_V/migration_shares) -- the
# production side (w_temp, pi_temp, L_0, A_0, kappa_0) never uses it -- so we override tau_mig
# and mu_stationary just for this call, then restore both (mirrors run_taumig_variation_scenario
# in the batch-sweep section above) so later cells in this section are unaffected.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = zeros(N, M, N, M)
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("Migration cost = 0 everywhere (frictionless mobility)")

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original;

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.227693389286969e-9)

Migration cost = 0 everywhere (frictionless mobility)
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.4228          0.2791          0.5953          0.7064          
  Region 2:  2.4228          0.7561          0.5561          0.2613          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9369          0.877           
  Region 2:  0.8193          0.9497          1.07            

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0            0.0             -0.0            -0.0            
  Region 2:  -0.0    

In [40]:
# Scenario: home-production value b drops -90% in every region, no other shock. Like tau_mig,
# b only enters through flow_utility_mkt_at -> stationary_V -> migration_shares (the baseline
# mu_stationary) -- solve_temporary_equilibrium/solve_temp_eq_hat never use it -- so we override
# b and mu_stationary just for this call, then restore both.
global b, mu_stationary
b_original = b
mu_stationary_original = mu_stationary

b = b .* 0.1
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("Home production value b -90% (all regions)")

b = b_original
mu_stationary = mu_stationary_original;

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 6.072278901925188e-9)

Home production value b -90% (all regions)
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.5305          0.5462          1.3096          1.6051          
  Region 2:  0.5305          1.7571          1.2074          0.5136          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9468          0.8648          
  Region 2:  0.7916          0.9623          1.0862          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0669         -0.0583         0.0339          0.0852          
  Region 2:  -0.0669         0.1

In [ ]:
# Convergence-lag comparison: same fundamentals shock (region 1, sector 1 productivity +50%,
# ramped in over periods 1-5, flat thereafter) run out to T=50, under three tau_mig regimes --
# a control at the current baseline tau_mig (=1 off-diagonal, as set by the reset cell above),
# plus a frictionless and a near-immobile case below. tau_mig only enters through
# mu_stationary, so the control needs no override -- it already reflects whatever tau_mig the
# reset cell left in place.
run_simple_scenario("Control: tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50);

In [ ]:
# Same shock, tau_mig = 0 everywhere (frictionless mobility) -- same convention as Scenario 9
# in the batch-sweep section above. Override tau_mig/mu_stationary just for this call, then
# restore both.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = zeros(N, M, N, M)
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("Low tau_mig=0 (frictionless), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original;

In [ ]:
# Same shock, tau_mig = 100000 everywhere off-diagonal (effectively no mobility) -- same
# convention as Scenario 10 in the batch-sweep section above. Override tau_mig/mu_stationary
# just for this call, then restore both.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = [(n == i && j == k) ? 0.0 : 100000.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("High tau_mig=100000 (effectively no mobility), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original;

In [ ]:
# Convergence-lag summary for the three tau_mig regimes above, pulled straight from
# simplified_scenario_runs (mirrors the Scenario 8-11 summary cell in the batch-sweep section).
summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
println(simplified_scenario_runs[in([
    "Control: tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50",
    "Low tau_mig=0 (frictionless), A[1,1] +50% ramped over t=1-5, T=50",
    "High tau_mig=100000 (effectively no mobility), A[1,1] +50% ramped over t=1-5, T=50",
]).(simplified_scenario_runs.label), summary_cols])

In [ ]:
# Convergence-lag comparison: same fundamentals shock (region 1, sector 1 productivity +50%,
# ramped in over periods 1-5, flat thereafter) run out to T=50, under three beta (discount
# factor) regimes -- a control at the current baseline beta (=0.95, as set by the reset cell
# above), plus a low and a high beta case below. Unlike tau_mig/b, beta enters the DYNAMICS
# directly (migration_shares_next and the backward u_dot step in solve_transition_path_hat both
# read the global beta at every period, not just the baseline mu_stationary), but the control
# still needs no override since it already reflects whatever beta the reset cell left in place.
run_simple_scenario("Control: beta=0.95 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50);

In [ ]:
# Same shock, beta = 0.5 (households barely value the future). beta feeds both mu_stationary
# (via stationary_V) AND the transition path itself (migration_shares_next, the backward u_dot
# step) -- so overriding the global beta before calling run_simple_scenario is enough to affect
# the whole run, not just the starting point. Restored afterward.
global beta, mu_stationary
beta_original = beta
mu_stationary_original = mu_stationary

beta = 0.5
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("Low beta=0.5, A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

beta = beta_original
mu_stationary = mu_stationary_original;

In [ ]:
# Same shock, beta = 0.99 (households very patient / forward-looking). Same override/restore
# pattern as the low-beta cell above.
global beta, mu_stationary
beta_original = beta
mu_stationary_original = mu_stationary

beta = 0.99
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("High beta=0.99, A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

beta = beta_original
mu_stationary = mu_stationary_original;

In [ ]:
# Convergence-lag summary for the three beta regimes above, pulled straight from
# simplified_scenario_runs (same style as the tau_mig summary cell above).
summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
println(simplified_scenario_runs[in([
    "Control: beta=0.95 (baseline), A[1,1] +50% ramped over t=1-5, T=50",
    "Low beta=0.5, A[1,1] +50% ramped over t=1-5, T=50",
    "High beta=0.99, A[1,1] +50% ramped over t=1-5, T=50",
]).(simplified_scenario_runs.label), summary_cols])

In [ ]:
# Convergence-lag comparison: same fundamentals shock (region 1, sector 1 productivity +50%,
# ramped in over periods 1-5, flat thereafter) run out to T=50, under three nu (migration taste-
# shock dispersion) regimes -- a "same" control at the current baseline nu (=1.0, as set by the
# reset cell above), plus a low and a high nu case below. Like beta, nu enters the DYNAMICS
# directly (migration_shares_next and the backward u_dot step both read the global nu at every
# period), but the control still needs no override since it already reflects whatever nu the
# reset cell left in place.
run_simple_scenario("Same: nu=1.0 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50);

In [ ]:
# Same shock, nu = 0.1 (migration decisions dominated by utility differences -- little
# idiosyncratic noise, very elastic/responsive migration). nu feeds both mu_stationary (via
# stationary_V) AND the transition path itself (migration_shares_next, the backward u_dot step)
# -- overriding the global nu before calling run_simple_scenario affects the whole run, not just
# the starting point. Restored afterward.
global nu, mu_stationary
nu_original = nu
mu_stationary_original = mu_stationary

nu = 0.1
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("Low nu=0.1, A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

nu = nu_original
mu_stationary = mu_stationary_original;

In [ ]:
# Same shock, nu = 5.0 (migration decisions dominated by idiosyncratic taste -- sluggish, barely
# responsive to utility gaps). Same override/restore pattern as the low-nu cell above.
global nu, mu_stationary
nu_original = nu
mu_stationary_original = mu_stationary

nu = 5.0
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

run_simple_scenario("High nu=5.0, A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)

nu = nu_original
mu_stationary = mu_stationary_original;

In [ ]:
# Convergence-lag summary for the three nu regimes above, pulled straight from
# simplified_scenario_runs (same style as the tau_mig and beta summary cells above).
summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
println(simplified_scenario_runs[in([
    "Same: nu=1.0 (baseline), A[1,1] +50% ramped over t=1-5, T=50",
    "Low nu=0.1, A[1,1] +50% ramped over t=1-5, T=50",
    "High nu=5.0, A[1,1] +50% ramped over t=1-5, T=50",
]).(simplified_scenario_runs.label), summary_cols])

In [ ]:
# tau_mig sweep at intermediate values, to check whether endog_convergence_lag is
# non-monotonic in mobility frictions: near-zero lag is expected at BOTH tau_mig=0 (migration
# reoptimizes essentially instantly) and tau_mig=100000 (migration is frozen, so there's
# nothing left to lag) -- see the discussion above. The interesting question is whether it's
# larger somewhere in between. tau_mig=0, 1, and 100000 were already run above (Low/Control/
# High); this fills in the gap. Same shock as before (region 1, sector 1 productivity +50%,
# ramped over t=1-5), T=50.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

intermediate_tau_values = [0.1, 5.0, 20.0, 100.0, 1000.0]
for cost_value in intermediate_tau_values
    tau_mig = [(n == i && j == k) ? 0.0 : cost_value for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
    mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))
    run_simple_scenario("tau_mig=$cost_value, A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50)
end

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original;

In [ ]:
# Full tau_mig scan summary, ordered ascending by tau_mig, combining the Low/Control/High runs
# from above with the intermediate sweep -- read endog_convergence_lag down this table to see
# whether it peaks away from the two boundaries (expected) or is flat/near-zero throughout
# (would be worth digging into further).
tau_scan_labels_to_tau = [
    ("Low tau_mig=0 (frictionless), A[1,1] +50% ramped over t=1-5, T=50", 0.0),
    ("tau_mig=0.1, A[1,1] +50% ramped over t=1-5, T=50", 0.1),
    ("Control: tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50", 1.0),
    ("tau_mig=5.0, A[1,1] +50% ramped over t=1-5, T=50", 5.0),
    ("tau_mig=20.0, A[1,1] +50% ramped over t=1-5, T=50", 20.0),
    ("tau_mig=100.0, A[1,1] +50% ramped over t=1-5, T=50", 100.0),
    ("tau_mig=1000.0, A[1,1] +50% ramped over t=1-5, T=50", 1000.0),
    ("High tau_mig=100000 (effectively no mobility), A[1,1] +50% ramped over t=1-5, T=50", 100000.0),
]

summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
tau_scan_summary = vcat([simplified_scenario_runs[findfirst(==(lbl), simplified_scenario_runs.label):findfirst(==(lbl), simplified_scenario_runs.label), summary_cols]
                          for (lbl, _) in tau_scan_labels_to_tau]...)
tau_scan_summary.tau_mig_offdiag = [t for (_, t) in tau_scan_labels_to_tau]
println(tau_scan_summary)

In [55]:
# It converged this time (outer_used=157) -- so this IS now a trustworthy equilibrium path, and
# the picture is different from the earlier crash: dev_w and dev_u decay reasonably fast (down to
# ~0.03/0.02 by t=100), but dev_L is the binding one -- it's still at 0.157 (15.7%) at t=100,
# decaying only ~10%/period near the tail (0.223 -> 0.157 from t=97 to t=100, a ratio of ~0.90^3).
# That's genuine geometric decay, just slow -- consistent with tau_mig=20 being exactly the
# "interior" friction regime we were looking for, where migration reallocates gradually over many
# periods rather than snapping instantly (tau_mig=0) or staying frozen (tau_mig=100000). At a
# ~0.90/period decay rate it'd take roughly another ~45 periods past t=100 to get dev_L under
# endog_tol=1e-3, so T=200 with more outer-iteration budget should be enough to actually see it
# cross the threshold and register a real endog_converged_period.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = [(n == i && j == k) ? 0.0 : 20.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

result_tau20 = run_simple_scenario("tau_mig=20.0 diagnostic rerun, A[1,1] +50% ramped over t=1-5, T=200, damp=0.2";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=200, max_outer=500, damp=0.2)

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original

println("\nouter loop converged: ", result_tau20.converged, "   outer_used: ", result_tau20.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau20.u_dot_path)
for p in 1:Tu
    dev_w = maximum(abs.(result_tau20.w_path[p+1] ./ result_tau20.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau20.L_path[p+1] ./ result_tau20.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau20.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

solve_transition_path_hat: reached max_outer=500 without converging (max u_dot change = 6278.065585374409)

tau_mig=20.0 diagnostic rerun, A[1,1] +50% ramped over t=1-5, T=200, damp=0.2
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.3479          0.3111          0.6185          0.7264          
  Region 2:  2.3479          0.788           0.5804          0.2799          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.904           0.8533          
  Region 2:  0.7782          0.9156          1.0326          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.6134          -0.2001         -0.2203         -0.2034    

In [ ]:
# Same diagnostic, tau_mig=12 -- how many outer iterations does this need to actually
# converge? Start with the same config that worked for tau_mig=20 at T=100 (damp=0.2,
# max_outer=1000, which took 157 iterations there).
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = [(n == i && j == k) ? 0.0 : 12.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

result_tau12 = run_simple_scenario("tau_mig=12.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=100, max_outer=1000, damp=0.2)

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original

println("\nouter loop converged: ", result_tau12.converged, "   outer_used: ", result_tau12.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau12.u_dot_path)
for p in 1:Tu
    dev_w = maximum(abs.(result_tau12.w_path[p+1] ./ result_tau12.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau12.L_path[p+1] ./ result_tau12.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau12.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

In [57]:
# Same diagnostic, tau_mig=17 -- tau_mig=12 converged cleanly at T=100/damp=0.2/max_outer=1000,
# tau_mig=20 needed T=200+ and still hadn't converged at max_outer=500 -- narrowing in on where
# it starts getting stiff.
global tau_mig, mu_stationary
tau_mig_original = tau_mig
mu_stationary_original = mu_stationary

tau_mig = [(n == i && j == k) ? 0.0 : 17.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
mu_stationary = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

result_tau17 = run_simple_scenario("tau_mig=17.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=100, max_outer=1000, damp=0.2)

tau_mig = tau_mig_original
mu_stationary = mu_stationary_original

println("\nouter loop converged: ", result_tau17.converged, "   outer_used: ", result_tau17.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau17.u_dot_path)
for p in 1:Tu
    dev_w = maximum(abs.(result_tau17.w_path[p+1] ./ result_tau17.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau17.L_path[p+1] ./ result_tau17.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau17.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

solve_transition_path_hat converged after 385 outer iterations (max u_dot change = 8.17973222488888e-9)

tau_mig=17.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.5651          0.2686          0.5375          0.6331          
  Region 2:  2.5651          0.6855          0.5037          0.2414          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9034          0.8505          
  Region 2:  0.7771          0.9152          1.0313          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.6518          -0.1857         -0.2365         -0.2369         
  R